# Entrenamiento del modelo clasificador de riesgo IA

En este notebook entrenamos un modelo baseline para clasificar textos en 4 categorías de riesgo del AI Act:
- Inaceptable
- Alto riesgo
- Riesgo limitado
- Riesgo mínimo

Pasos:
1. Carga de datos (train, validation y test)
2. Vectorización con TF-IDF (bigramas, sublinear_tf)
3. Encoding de features estructuradas (category, context, longitud, num_articles)
4. Combinación de features y entrenamiento (LogisticRegression + class_weight='balanced')
5. Evaluación en validación
6. Guardar artefactos (modelo + vectorizador + encoder)

In [1]:
%load_ext autoreload
%autoreload 2

## 1. Carga de datos

In [2]:
import pandas as pd

train_df = pd.read_csv("data/processed/train.csv")
val_df   = pd.read_csv("data/processed/validation.csv")
test_df  = pd.read_csv("data/processed/test.csv")

X_train_text = train_df["text_final"]
X_val_text   = val_df["text_final"]
X_test_text  = test_df["text_final"]

y_train = train_df["etiqueta"]
y_val   = val_df["etiqueta"]
y_test  = test_df["etiqueta"]

# Columnas estructuradas disponibles tras el preprocesado
CAT_COLS = ["category", "context"]
NUM_COLS = ["longitud", "num_articles"]

print(f"Train: {len(train_df)} muestras | Columnas: {list(train_df.columns)}")
print(f"Validation: {len(val_df)} muestras")
print(f"Test: {len(test_df)} muestras")

Train: 210 muestras | Columnas: ['text_final', 'category', 'context', 'longitud', 'num_articles', 'etiqueta']
Validation: 45 muestras
Test: 45 muestras


## 2. Vectorización TF-IDF

In [3]:
from functions import crear_tfidf

tfidf, X_train_tfidf, X_val_tfidf, X_test_tfidf = crear_tfidf(
    X_train_text, X_val_text, X_test_text,
    max_features=5000,
    ngram_range=(1, 2),
)

OSError: [WinError 1114] Error en una rutina de inicialización de biblioteca de vínculos dinámicos (DLL). Error loading "c:\Users\rammu\anaconda3\envs\venv_proyecto\Lib\site-packages\torch\lib\c10.dll" or one of its dependencies.

## 3. Encoding de features estructuradas

In [ ]:
from sklearn.preprocessing import OneHotEncoder
from scipy.sparse import csr_matrix, hstack

# One-hot encoding de category y context.
# IMPORTANTE: fit solo en train para evitar data leakage hacia val/test.
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
cat_train = ohe.fit_transform(train_df[CAT_COLS])
cat_val   = ohe.transform(val_df[CAT_COLS])
cat_test  = ohe.transform(test_df[CAT_COLS])

# Features numéricas como sparse matrix
num_train = csr_matrix(train_df[NUM_COLS].values.astype(float))
num_val   = csr_matrix(val_df[NUM_COLS].values.astype(float))
num_test  = csr_matrix(test_df[NUM_COLS].values.astype(float))

# Combinar TF-IDF + one-hot + numéricas
X_train_final = hstack([X_train_tfidf, cat_train, num_train])
X_val_final   = hstack([X_val_tfidf,   cat_val,   num_val])
X_test_final  = hstack([X_test_tfidf,  cat_test,  num_test])

print(f"Features TF-IDF:       {X_train_tfidf.shape[1]}")
print(f"Features categóricas:  {cat_train.shape[1]}  (category: {len(ohe.categories_[0])} cats, context: {len(ohe.categories_[1])} cats)")
print(f"Features numéricas:    {len(NUM_COLS)}  ({NUM_COLS})")
print(f"Total features final:  {X_train_final.shape[1]}")

Features TF-IDF:       3773
Features categóricas:  22  (category: 9 cats, context: 13 cats)
Features numéricas:    2  (['longitud', 'num_articles'])
Total features final:  3797


## 4. Entrenamiento del modelo (LogisticRegression + class_weight='balanced')

In [ ]:
from functions import entrenar_modelo_baseline

# class_weight='balanced' compensa el desbalanceo 6.5× (inaceptable:131 vs riesgo_limitado:20)
modelo = entrenar_modelo_baseline(
    X_train_final, y_train,
    X_val_final,   y_val,
    class_weight="balanced",
)

=== Resultados en VALIDACIÓN ===

                 precision    recall  f1-score   support

    alto_riesgo       0.29      0.57      0.38         7
    inaceptable       0.67      0.50      0.57        20
riesgo_limitado       0.50      0.33      0.40         3
  riesgo_minimo       0.64      0.60      0.62        15

       accuracy                           0.53        45
      macro avg       0.52      0.50      0.49        45
   weighted avg       0.59      0.53      0.55        45

F1-score macro (validación): 0.4933


## 5. Guardar artefactos

In [ ]:
import joblib
import os

output_dir = "model"
os.makedirs(output_dir, exist_ok=True)

joblib.dump(modelo, os.path.join(output_dir, "modelo_baseline.joblib"))
joblib.dump(tfidf,  os.path.join(output_dir, "tfidf_vectorizer.joblib"))
joblib.dump(ohe,    os.path.join(output_dir, "ohe_encoder.joblib"))

print("Modelo baseline guardado en:  model/modelo_baseline.joblib")
print("Vectorizador TF-IDF guardado: model/tfidf_vectorizer.joblib")
print("OHE encoder guardado en:      model/ohe_encoder.joblib")

Modelo baseline guardado en:  model/modelo_baseline.joblib
Vectorizador TF-IDF guardado: model/tfidf_vectorizer.joblib
OHE encoder guardado en:      model/ohe_encoder.joblib


In [ ]:
# ── MLflow (solo falla esta celda si el servidor no está disponible) ──
from functions import log_mlflow_safe
from sklearn.metrics import f1_score, accuracy_score

y_val_pred = modelo.predict(X_val_final)

try:
    log_mlflow_safe(
        run_name="exp0_baseline",
        params={
            "model_type":         "LogisticRegression",
            "model_max_iter":     2000,
            "class_weight":       "balanced",
            "tfidf_max_features": 5000,
            "tfidf_ngram_range":  "(1, 2)",
            "tfidf_sublinear_tf": True,
            "extra_features":     "category,context,longitud,num_articles",
            "n_features_total":   X_train_final.shape[1],
        },
        metrics={
            "val_f1_macro": round(f1_score(y_val, y_val_pred, average="macro"), 4),
            "val_accuracy": round(accuracy_score(y_val, y_val_pred), 4),
        },
        artifacts=[
            "model/modelo_baseline.joblib",
            "model/tfidf_vectorizer.joblib",
            "model/ohe_encoder.joblib",
        ],
        tags={"experimento": "0", "features": "tfidf+category+context+longitud+num_articles"},
    )
    print("✓ Exp 0 (baseline) registrado en MLflow")
except Exception as e:
    print(f"⚠ MLflow no disponible: {e}")

Password obtenida desde variable de entorno local.
MLflow configurado correctamente → https://34.244.146.100
⚠ MLflow no disponible: API request to https://34.244.146.100/api/2.0/mlflow/experiments/get-by-name failed with exception HTTPSConnectionPool(host='34.244.146.100', port=443): Max retries exceeded with url: /api/2.0/mlflow/experiments/get-by-name?experiment_name=clasificador_riesgo_ia_artificial (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self-signed certificate (_ssl.c:1010)')))
